# Data Preparation Note: TORD v4 (Token Offering Dataset)

**Project context:** Master’s thesis (Finance + Analytics) using the TORD v4 dataset of token offerings.  
**Goal:** Provide a submission-ready, transparent description of the data preparation steps, decisions, and outputs created during cleaning.  
**Software:** Python 3.13.3 (pandas, NumPy).  
**Input file:** `tord_v4.csv`  
**Outputs folder:** `outputs_tord/`

---

## Outputs produced

### `outputs_tord/tord_clean_min.csv`
Cleaned offering-level dataset used for modelling and for merging to CoinMarketCap OHLCV time series.  
Includes both cleaned original variables and engineered/parsed variables (notably for `min_investment`).

### `outputs_tord/tord_audit_before.csv`
Column-level audit of the raw dataset before cleaning:
- pandas dtype
- missingness percentage
- number of unique values
- example values (up to 3)

### `outputs_tord/tord_audit_after.csv`
Column-level audit after cleaning (same structure as the “before” audit).

### `outputs_tord/tord_clean_log.txt`
Text log capturing:
- input file path and initial shape
- schema-missing columns (if any)
- deduplication results
- output file paths and final shape

---

## 1. Why a “minimal but robust” cleaning strategy?

The project will merge TORD offering characteristics to **post-offering market data** (CoinMarketCap OHLCV). For that pipeline to work reliably, the offering dataset must be cleaned in a way that:

1. Preserves information (avoid unnecessary row drops).
2. Standardizes types (dates, numeric values, dummy variables).
3. Supports reproducibility (audits and a log file).
4. Maintains interpretability (minimal and defensible feature engineering).

---

## 2. Schema-driven approach

Cleaning decisions were guided by variable roles (“schema”). Each variable was assigned a role:

- **ID:** `id`
- **Strings/categories:** `name`, `country`, `restricted_areas`, `accepting`, links (e.g., `website`)
- **Ticker:** `token`
- **Dates:** `ico_start`, `ico_end`, `pre_ico_start`, `pre_ico_end`
- **Numeric:** `price_usd`, `raised_usd`, `teamsize`, `rating`, `Coinmarketcap_identifier`, `sold_tokens`, `pre_ico_price_usd`, `token_for_sale` (coerced where possible)
- **Dummies (0/1):** `is_ico`, `is_ieo`, `is_sto`, `whitelist`, `kyc`, `bonus`, `bounty`, `mvp`, `ERC20`, `platform`
- **Percent or fraction:** `distributed_in_ico` (stored as either 0–100 or 0–1; normalized to fraction)
- **Free-text numeric-with-unit field (parsed into structured features):** `min_investment`

---

## 3. Cleaning steps performed

### 3.1 Column name trimming
Column names were trimmed for leading/trailing whitespace.

### 3.2 Missing-value normalization (string fields)
Common missing tokens were converted to missing values (`pd.NA`) for string-like columns.

Normalized tokens include:  
`""`, `"na"`, `"n/a"`, `"null"`, `"none"`, `"nan"`, `"-"`, `"--"`, `"?"`, `"missing"`, `"no"`, `"nil"`.

### 3.3 Ticker normalization (`token`)
Ticker symbols were uppercased and whitespace was removed.

### 3.4 Date parsing
Date fields were parsed using `pd.to_datetime(..., errors="coerce")`. Invalid dates become missing.

### 3.5 Dummy variable coercion (0/1)
Dummy fields were mapped to 0/1 from common text forms (`yes/no`, `true/false`, `y/n`, `1/0`, etc.) and clipped to [0, 1].

### 3.6 Numeric coercion
Numeric fields were coerced from strings using a robust “first-number extraction” approach:
- handles parentheses negatives (e.g., `(123)` → `-123`)
- handles decimal commas (e.g., `0,1` → `0.1`)
- removes thousands separators (e.g., `1,000` or `10.000` → `1000` / `10000`)
- strips common currency symbols/codes and percent signs
- extracts the first numeric substring and converts with `pd.to_numeric(..., errors="coerce")`

### 3.7 Percent-to-fraction normalization for `distributed_in_ico`
A normalized variable `distributed_in_ico_frac` was created:
- If `distributed_in_ico > 1`, interpret as percent and divide by 100.
- Else treat as already a fraction.

### 3.8 Parsing `min_investment` (free-text amounts with units)
`min_investment` is not consistently numeric (examples include `50 USD`, `0.1 ETH`, `1 ETH or 0.1 BTC`, `0,1 ETH`, `10.000 USD`, `$50 USD`).  
Therefore the original column is preserved as text and parsed into structured fields:

- `min_investment_raw`: original text
- `min_inv_clean`: normalized formatting (decimal comma → dot; thousands separators removed; spacing fixed; separators normalized)
- `min_inv_pairs`: JSON list of extracted `{value, unit}` pairs (supports fiat codes and token symbols; `$`→USD, `€`→EUR, `£`→GBP; known unit typo fix `GBR`→`GBP`)
- `min_inv_min_value`, `min_inv_min_unit`: the **minimum numeric value** among extracted pairs (conservative proxy; note values are not comparable across units)
- `min_inv_usd_value`: the minimum value **among USD pairs only** (no conversion from ETH/BTC/etc.)
- flags:  
  - `flag_min_inv_multi`: multiple options/pairs detected (e.g., “or”, “/”, multiple extracted pairs)  
  - `flag_min_inv_ratio`: ratio/conditional pricing patterns detected (e.g., “ICO”, “Pre-ICO”, “per”, “=”)  
  - `flag_min_inv_equals`: explicit equals sign detected  
  - `flag_min_inv_missinglike`: missing-like tokens (e.g., `no`, `0`, `-`)  

### 3.9 Minimal feature engineering
The following variables were added:
- `ico_duration_days = ico_end - ico_start` (days)
- `has_presale = 1{pre_ico_start not missing}`
- link presence indicators: `has_whitepaper`, `has_github`, `has_linkedin`, `has_website`

### 3.10 Deduplication
If present, duplicates were removed by `id` (keeping the first occurrence). Otherwise, exact duplicates were removed.

### 3.11 Sanity flags (no row drops)
Observations were not removed; instead, flags were added:
- `flag_suspicious_ico_start = 1{ico_start < 2009-01-01}`
- `flag_negative_price = 1{price_usd < 0}`
- `flag_negative_raised = 1{raised_usd < 0}`

---

## 4. Known limitations and next steps

### Limitations
- Missingness remains substantial for some offering characteristics (e.g., `raised_usd`, `min_investment`).
- `min_inv_min_value` is a conservative proxy and **not directly comparable across units** (ETH vs USD etc.).
- TORD does not include post-offering prices/returns/liquidity; it is offering-level.

### Next steps
- Merge TORD with CoinMarketCap daily OHLCV using `Coinmarketcap_identifier`.
- Define post-offering start date (`t0`) based on first day with non-trivial price/volume.
- Construct investment-relevant outcomes (returns, drawdowns, crash risk, liquidity proxies) for econometric and ML modelling.

---

## 5. Reproducibility checklist
1. Place `tord_v4.csv` in the project folder.
2. Run the Python cleaning script (Python 3.13.3).
3. Confirm outputs in `outputs_tord/`:
   - `tord_clean_min.csv`
   - `tord_audit_before.csv`
   - `tord_audit_after.csv`
   - `tord_clean_log.txt`
4. Review the audit files to verify types and missingness.


In [ ]:
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import re
import json

# ===== CONFIG =====
RAW_DIR       = Path("../raw")
PROCESSED_DIR = Path("../processed")

INPUT            = RAW_DIR       / "tord_v4.csv"
OUT_CLEAN        = PROCESSED_DIR / "tord_clean_min.csv"
OUT_AUDIT_BEFORE = PROCESSED_DIR / "tord_audit_before.csv"
OUT_AUDIT_AFTER  = PROCESSED_DIR / "tord_audit_after.csv"
OUT_LOG          = PROCESSED_DIR / "tord_clean_log.txt"

# ===== SCHEMA =====
SCHEMA = {
    "id": "id",
    "name": "str",
    "token": "ticker",
    "country": "str",
    "is_ico": "dummy", "is_ieo": "dummy", "is_sto": "dummy",
    "ico_start": "date", "ico_end": "date",
    "price_usd": "num", "raised_usd": "num",
    "distributed_in_ico": "pct",
    "sold_tokens": "num", "token_for_sale": "num",
    "whitelist": "dummy", "kyc": "dummy", "bonus": "dummy",
    "restricted_areas": "str",
    "min_investment": "str",
    "bounty": "dummy", "mvp": "dummy",
    "pre_ico_start": "date", "pre_ico_end": "date",
    "pre_ico_price_usd": "num",
    "platform": "dummy",
    "accepting": "str",
    "link_white_paper": "str", "linkedin_link": "str", "github_link": "str", "website": "str",
    "rating": "num", "teamsize": "num",
    "Coinmarketcap_identifier": "num",
    "ERC20": "dummy",}

MISSING = {"", " ", "na", "n/a", "null", "none", "nan", "-", "--", "?", "missing", "no", "nil"}
BOOL = {"1": 1, "0": 0, "yes": 1, "no": 0, "true": 1, "false": 0,"y": 1, "n": 0, "t": 1, "f": 0,"available": 1, "not available": 0 }
NONVALUE_TOKENS = {"no", "none", "n/a", "na", "null", "nil", "0", "zero", "-"}


def log(msg: str) -> None:
    stamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with OUT_LOG.open("a", encoding="utf-8") as f:
        f.write(f"[{stamp}] {msg}\n")


def audit(df: pd.DataFrame, label: str) -> pd.DataFrame:
    return pd.DataFrame({
        "dataset": label,
        "column": df.columns,
        "dtype": [str(df[c].dtype) for c in df.columns],
        "missing_%": [df[c].isna().mean() * 100 for c in df.columns],
        "n_unique": [df[c].nunique(dropna=True) for c in df.columns],
        "examples": [" | ".join(df[c].dropna().astype(str).head(3).tolist()) for c in df.columns],
    }).sort_values(["missing_%", "n_unique"], ascending=[False, False])


def norm_missing(s: pd.Series) -> pd.Series:
    x = s.astype("string").str.strip()
    return x.where(~x.str.lower().isin(MISSING), pd.NA)

def normalize_min_inv_text(x: pd.Series) -> pd.Series:
    s = x.astype("string")

    # Normalize NBSP and weird whitespace
    s = s.str.replace("\u00A0", " ", regex=False).str.strip()

    # Normalize common currency symbols to units (keep symbols too as a fallback)
    # We'll still parse $ and € in regex below.
    s = s.str.replace("usd", "USD", case=False, regex=False)
    s = s.str.replace("eur", "EUR", case=False, regex=False)
    s = s.str.replace("chf", "CHF", case=False, regex=False)
    s = s.str.replace("gbp", "GBP", case=False, regex=False)

    # Convert decimal comma to dot when between digits: 0,1 -> 0.1
    s = s.str.replace(r"(?<=\d),(?=\d)", ".", regex=True)

    # Convert EU thousands like 10.000 -> 10000 (only when dot separates 3 digits)
    s = s.str.replace(r"(\d)\.(\d{3})(\D|$)", r"\1\2\3", regex=True)

    # Remove thousands commas: 1,000 -> 1000
    s = s.str.replace(",", "", regex=False)

    # Ensure space between number and letters: 0.4ETH -> 0.4 ETH
    s = s.str.replace(r"(\d)([A-Za-z])", r"\1 \2", regex=True)

    # Ensure separation in glued sequences: 0.01 BTC0.1 ETH -> 0.01 BTC 0.1 ETH
    s = s.str.replace(r"([A-Za-z]{2,10})(\d)", r"\1 \2", regex=True)

    # Normalize separators
    s = s.str.replace(r"[;/|]", " / ", regex=True)   # keep slash meaningful
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()

    return s


_PAIR_REGEX = re.compile(
    r"(?P<value>[-+]?(?:\d+\.\d+|\d+|\.\d+)(?:e[-+]?\d+)?)\s*(?P<unit>[A-Za-z]{2,10}|\$|€|£)",
    flags=re.IGNORECASE
)
UNIT_FIX = {"GBR": "GBP"}

def extract_pairs(text: str) -> list[dict]:
    """
    Extract all (value, unit) pairs from a single string.
    Units can be token symbols or fiat codes; $ € £ are converted to USD/EUR/GBP.
    """
    if text is None:
        return []
    t = str(text).strip()
    if not t:
        return []
    tl = t.lower().strip()
    if tl in NONVALUE_TOKENS:
        return []

    pairs = []
    for m in _PAIR_REGEX.finditer(t):
        v = m.group("value")
        u = m.group("unit").upper()

        # symbol -> ISO code
        if u == "$":
            u = "USD"
        elif u == "€":
            u = "EUR"
        elif u == "£":
            u = "GBP"
        u = UNIT_FIX.get(u, u)

        try:
            val = float(v)
        except ValueError:
            continue

        # Drop obviously invalid
        if np.isnan(val):
            continue

        pairs.append({"value": val, "unit": u})

    return pairs


def summarize_min_threshold(pairs: list[dict]) -> tuple[float | None, str | None]:
    """
    Choose a single representative 'minimum investment threshold' from all pairs.
    Rule: pick the smallest numeric value across all extracted pairs.
    Note: This is only meaningful within-unit, but it still gives a conservative threshold proxy.
    """
    if not pairs:
        return None, None
    # choose minimum by numeric value
    best = min(pairs, key=lambda d: d["value"])
    return best["value"], best["unit"]


def parse_min_investment_column(s: pd.Series) -> pd.DataFrame:
    """
    Full parsing pipeline returning:
      - min_investment_raw
      - min_inv_clean
      - min_inv_pairs (json string)
      - min_inv_min_value, min_inv_min_unit
      - min_inv_usd_value (if any USD appears; choose smallest USD)
      - flags for messy patterns
    """
    raw = s.astype("string")
    clean = normalize_min_inv_text(raw)

    flag_ratio = clean.str.contains(r"pre-?ico|ico|\bper\b|=", case=False, regex=True).fillna(False).astype(int)
    flag_equals = clean.str.contains(r"=", case=False, regex=True).fillna(False)
    flag_missinglike = clean.str.lower().isin(NONVALUE_TOKENS) | clean.isna() | (clean.str.len() == 0)

    pairs_list = clean.apply(lambda x: extract_pairs(x) if pd.notna(x) else [])

    flag_multi_text = clean.str.contains(r"\bor\b|/|,", case=False, regex=True).fillna(False)
    flag_multi_pairs = pairs_list.apply(lambda lst: len(lst) > 1)

    flag_multi = (flag_multi_text | flag_multi_pairs).astype(int)

    # Summaries
    min_val_unit = pairs_list.apply(summarize_min_threshold)
    min_val = min_val_unit.apply(lambda t: t[0]).astype("Float64")
    min_unit = min_val_unit.apply(lambda t: t[1]).astype("string")

    # USD-only numeric (do NOT convert ETH/BTC etc)
    def min_usd(pairs: list[dict]) -> float | None:
        usd_vals = [p["value"] for p in pairs if p["unit"] == "USD"]
        if not usd_vals:
            return None
        return float(min(usd_vals))

    usd_val = pairs_list.apply(min_usd).astype("Float64")

    # Serialize pairs to JSON string (keeps all information)
    pairs_json = pairs_list.apply(lambda lst: json.dumps(lst) if lst else "").astype("string")

    return pd.DataFrame({
        "min_investment_raw": raw,
        "min_inv_clean": clean,
        "min_inv_pairs": pairs_json,
        "min_inv_min_value": min_val,
        "min_inv_min_unit": min_unit,
        "min_inv_usd_value": usd_val,
        "flag_min_inv_multi": flag_multi,
        "flag_min_inv_ratio": flag_ratio,
        "flag_min_inv_equals": flag_equals.astype(int),
        "flag_min_inv_missinglike": flag_missinglike.fillna(False).astype(int),
    })

def to_num(s: pd.Series) -> pd.Series:
    """
    Robust numeric coercion for columns that are supposed to be numeric.
    Extracts the first number from a string and converts it to float.
    Handles commas/dots, currency symbols, %, parentheses negatives, scientific notation.
    """
    x = s.astype("string").str.strip()
    x = x.where(~x.str.lower().isin(MISSING), pd.NA)
    x = x.str.replace(r"^\((.*)\)$", r"-\1", regex=True)
    x = x.str.replace("\u00A0", " ", regex=False)
    x = x.str.replace(r"(?<=\d),(?=\d)", ".", regex=True)
    x = x.str.replace(r"(\d)\.(\d{3})(\D|$)", r"\1\2\3", regex=True)
    x = x.str.replace(",", "", regex=False)
    x = x.str.replace(r"(?i)(usd|eur|gbp|chf)", "", regex=True)
    x = x.str.replace(r"[\$,€£%]", "", regex=True)
    x = x.str.replace(r"\s+", " ", regex=True).str.strip()
    num = x.str.extract(r"([-+]?\d*\.?\d+(?:e[-+]?\d+)?)", expand=False)

    return pd.to_numeric(num, errors="coerce")


def to_dummy(s: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors="coerce").clip(0, 1)

    x = s.astype("string").str.strip().str.lower()
    mapped = x.map(BOOL).astype("Float64")

    m = mapped.isna() & x.notna() & (x != "")
    if m.any():
        mapped.loc[m] = pd.to_numeric(x[m], errors="coerce").astype("Float64")

    return pd.to_numeric(mapped, errors="coerce").clip(0, 1)


def main():
    # reset log
    OUT_LOG.write_text("", encoding="utf-8")
    log(f"Reading input: {INPUT.resolve()}")

    df = pd.read_csv(INPUT, encoding_errors="replace")
    df.columns = [c.strip() for c in df.columns]
    log(f"Loaded shape: {df.shape}")

    # audit BEFORE
    audit(df, "before").to_csv(OUT_AUDIT_BEFORE, index=False)
    log(f"Wrote audit before: {OUT_AUDIT_BEFORE.resolve()}")

    # normalize missing tokens in string-like columns
    for c in df.columns:
        if df[c].dtype == "object" or pd.api.types.is_string_dtype(df[c]):
            df[c] = norm_missing(df[c])

    # schema-based cleaning
    for col, role in SCHEMA.items():
        if col not in df.columns:
            log(f"[WARN] missing column: {col}")
            continue

        if role in {"str", "id"}:
            df[col] = df[col].astype("string").str.strip()

        elif role == "ticker":
            df[col] = df[col].astype("string").str.strip().str.upper().str.replace(r"\s+", "", regex=True)

        elif role == "date":
            df[col] = pd.to_datetime(df[col], errors="coerce")

        elif role == "dummy":
            df[col] = to_dummy(df[col])

        elif role == "num":
            df[col] = df[col] if pd.api.types.is_numeric_dtype(df[col]) else to_num(df[col])
            df[col] = pd.to_numeric(df[col], errors="coerce")

        elif role == "pct":
            base = df[col] if pd.api.types.is_numeric_dtype(df[col]) else to_num(df[col])
            base = pd.to_numeric(base, errors="coerce")
            df[col] = base
            df[col + "_frac"] = np.where(base > 1, base / 100.0, base)

    if "min_investment" in df.columns:
        parsed = parse_min_investment_column(df["min_investment"])
        df = pd.concat([df, parsed], axis=1)
        log("Parsed min_investment into structured min_inv_* columns.")
        cov_pairs = (df["min_inv_pairs"].astype("string").str.len() > 0).mean() * 100
        cov_usd = df["min_inv_usd_value"].notna().mean() * 100
        print(f"min_investment parsed coverage (any pairs): {cov_pairs:.2f}%")
        print(f"min_investment USD coverage: {cov_usd:.2f}%")
        print(df["min_inv_min_unit"].value_counts(dropna=True).head(15))

    # engineered features
    if {"ico_start", "ico_end"}.issubset(df.columns):
        df["ico_duration_days"] = (df["ico_end"] - df["ico_start"]).dt.days

    if "pre_ico_start" in df.columns:
        df["has_presale"] = df["pre_ico_start"].notna().astype(int)

    for flag, c in [
        ("has_whitepaper", "link_white_paper"),
        ("has_github", "github_link"),
        ("has_linkedin", "linkedin_link"),
        ("has_website", "website"),
    ]:
        if c in df.columns:
            df[flag] = (df[c].notna() & (df[c].astype("string").str.len() > 0)).astype(int)

    # deduplication
    before_n = len(df)
    if "id" in df.columns:
        df = df.drop_duplicates(subset=["id"], keep="first")
        log(f"Deduplicated on id: {before_n} -> {len(df)}")
    else:
        df = df.drop_duplicates()
        log(f"Deduplicated exact rows: {before_n} -> {len(df)}")

    # sanity flags (no row drops)
    if "ico_start" in df.columns:
        df["flag_suspicious_ico_start"] = (df["ico_start"] < pd.Timestamp("2009-01-01")).fillna(False).astype(int)
    if "price_usd" in df.columns:
        df["flag_negative_price"] = (df["price_usd"] < 0).fillna(False).astype(int)
    if "raised_usd" in df.columns:
        df["flag_negative_raised"] = (df["raised_usd"] < 0).fillna(False).astype(int)

    # audit AFTER
    audit(df, "after").to_csv(OUT_AUDIT_AFTER, index=False)
    log(f"Wrote audit after: {OUT_AUDIT_AFTER.resolve()}")

    # save cleaned data
    df.to_csv(OUT_CLEAN, index=False)
    log(f"Wrote cleaned file: {OUT_CLEAN.resolve()}")
    log(f"Final shape: {df.shape}")

    print("Saved:", OUT_CLEAN)
    print("Audit before:", OUT_AUDIT_BEFORE)
    print("Audit after:", OUT_AUDIT_AFTER)
    print("Log:", OUT_LOG)
    print("Rows/Cols:", df.shape)


if __name__ == "__main__":
    main()


min_investment parsed coverage (any pairs): 18.96%
min_investment USD coverage: 5.66%
min_inv_min_unit
ETH    1035
USD     609
EUR      61
BTC      42
NEO       6
CHF       6
XLM       3
GBP       3
KVT       2
CCX       2
ZNX       2
DCO       2
EXO       2
GAS       2
ROC       2
Name: count, dtype: Int64
Saved: outputs_tord\tord_clean_min.csv
Audit before: outputs_tord\tord_audit_before.csv
Audit after: outputs_tord\tord_audit_after.csv
Log: outputs_tord\tord_clean_log.txt
Rows/Cols: (10807, 56)


In [1]:
import pandas as pd
from pathlib import Path

TORD_PATH = Path(r"C:\Users\paolina.gocheva\OneDrive - Solactive AG\Desktop\me\datasets\final data\tord_clean_min.csv")

tord = pd.read_csv(TORD_PATH)

tord = tord[tord["Coinmarketcap_identifier"].notna()].copy()
tord["cmc_id"] = pd.to_numeric(tord["Coinmarketcap_identifier"], errors="coerce")
tord = tord[tord["cmc_id"].notna()].copy()
tord["cmc_id"] = tord["cmc_id"].astype(int)

counts = tord.groupby("cmc_id").size()

n_tokens_total = counts.shape[0]
n_tokens_single = (counts == 1).sum()
n_tokens_multi = (counts > 1).sum()
share_multi = n_tokens_multi / n_tokens_total * 100

print("Total unique tokens:", n_tokens_total)
print("Tokens with exactly one TORD row:", n_tokens_single)
print("Tokens with multiple TORD rows:", n_tokens_multi)
print("Share with multiple rows: {:.2f}%".format(share_multi))
print("Max number of TORD rows for one token:", counts.max())

Total unique tokens: 3428
Tokens with exactly one TORD row: 3387
Tokens with multiple TORD rows: 41
Share with multiple rows: 1.20%
Max number of TORD rows for one token: 2
